# Suite2p segmentation

Two ways to get a `Suite2pRois` object:

1. **Run detection** on an in-memory motion-corrected imaging object with `detect_rois_suite2p`.
2. **Reload** a pre-computed suite2p output folder via `Suite2pRois(folder)`.

Either way, the result is a `Suite2pRois` you can pass to `RoiAnalyzer`, plot with the widgets, or extract traces from.

In [ ]:
import numpy as np

import photon_mosaic as pm
import photon_mosaic.widgets as pw
from photon_mosaic.extractors import Suite2pImaging, Suite2pRois
from photon_mosaic.segmentation import (
    Suite2pSegmentationSettings,
    detect_rois_suite2p,
)

%matplotlib widget

## 1. Detect ROIs on a registered imaging object

Run suite2p detection on whatever motion-corrected `BaseImaging` you have in memory — `NumpyImaging`, `ZarrImaging`, or the output of `photon_mosaic.preprocessing.suite2p_registration.run_suite2p_registration(...)`.

Frames are streamed lazily through the imaging's epoch API, so the full raw movie is never materialised.

In [ ]:
# Stand-in for a real registered movie. Swap this for your own BaseImaging.
imaging = pm.generate_random_imaging(num_frames=2000, sampling_frequency=30.0)
imaging

In [ ]:
settings = Suite2pSegmentationSettings(
    diameter=[12.0, 12.0],
    tau=1.5,  # GCaMP6s; use ~0.7 for GCaMP6f
    fs=imaging.sampling_frequency,
    algorithm="sparsery",  # or "sourcery", "cellpose"
)

In [ ]:
# scope="all_epochs" (default) merges all epochs and returns one Suite2pRois.
rois = detect_rois_suite2p(imaging, settings=settings)
rois

In [ ]:
# detect_rois_suite2p auto-registers the imaging it ran on.
assert rois.imaging is imaging
rois.has_imaging()

### Per-epoch detection

When you'd rather have one ROI set per epoch (instead of merging them), pass `scope="per_epoch"`. The result is a `Suite2pEpochSegmentations` mapping `epoch_index -> Suite2pRois`.

In [ ]:
rois_by_epoch = detect_rois_suite2p(imaging, scope="per_epoch", settings=settings)
for epoch_idx, rois_ep in rois_by_epoch.items():
    print(f"epoch {epoch_idx}: {rois_ep.get_num_rois()} ROIs")

### Restricting epochs, pixel range, and bad frames

* `epoch_indices` — only run detection on a subset of epochs.
* `yrange` / `xrange` — crop the field of view before detection.
* `badframes` — boolean mask of frames to exclude (one flat array, or one per epoch).

In [ ]:
n0 = imaging.get_num_samples(segment_index=0)
bad0 = np.zeros(n0, dtype=bool)
bad0[100:120] = True  # drop a noisy stretch

rois_subset = detect_rois_suite2p(
    imaging,
    epoch_indices=[0],
    badframes=[bad0],
    yrange=[10, imaging.shape[0] - 10],
    xrange=[10, imaging.shape[1] - 10],
    settings=settings,
)
rois_subset.get_num_rois()

## 2. Reload a pre-computed suite2p folder

If suite2p has already been run (either by `photon_mosaic` or by an upstream pipeline), you can skip detection entirely:

* `Suite2pRois(folder)` reads `stat.npy`, `iscell.npy`, and `ops.npy` for the ROI side.
* `Suite2pImaging(folder)` reads `data.bin` + `ops.npy` for the registered movie.
* Pair them with `register_imaging` so downstream code can pull frames per ROI.

In [ ]:
folder = "/path/to/session/suite2p/plane0"  # <- replace with your folder

rois = Suite2pRois(folder)
imaging = Suite2pImaging(folder)
rois.register_imaging(imaging)

rois

In [ ]:
# Per-ROI properties pulled straight from stat.npy / iscell.npy.
print("num rois:", rois.get_num_rois())
print("is cell:", rois.get_property("iscell"))
print("is cell prob:", rois.get_property("iscell_probability"))

If your registered movie lives somewhere other than the suite2p folder (a Zarr store, a NWB file, a `BinaryFolderImaging`, ...), just register that imaging instead of `Suite2pImaging`:

In [ ]:
# rois = Suite2pRois(folder)
# rois.register_imaging(my_zarr_imaging)  # any BaseImaging with matching shape & fs

## 3. Downstream: plotting

Once `rois.imaging` is set (auto via `detect_rois_suite2p`, or manual via `register_imaging`), the rois can be plotted

In [ ]:
pw.plot_rois(rois, backend="ipywidgets", width_cm=20)

## Appendix: `Suite2pRois.from_stat` for in-memory `stat`

If you ran suite2p directly and already have a `stat` list in hand (no folder, no detection wrapper), build the ROIs without going through `detect_rois_suite2p`:

In [ ]:
# stat is a list[dict] from suite2p (each must have ypix, xpix, lam)
# rois = Suite2pRois.from_stat(
#     stats=stat,
#     shape=(Ly, Lx, 1),
#     sampling_frequency=fs,
# )
# rois.register_imaging(imaging)